In [1]:
import pandas as pd
import numpy as np
import os
import glob

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
cic_path = "../data/CICIDS2017/raw"

cic_files = glob.glob(os.path.join(cic_path, "*.csv"))

print("Number of CSV files:", len(cic_files))

for file in cic_files:
    print(os.path.basename(file))

Number of CSV files: 8
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Friday-WorkingHours-Morning.pcap_ISCX.csv
Monday-WorkingHours.pcap_ISCX.csv
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Wednesday-workingHours.pcap_ISCX.csv


In [3]:
# Load the Monday dataset for cleaning demonstration

monday_file = os.path.join(
    cic_path,
    "Monday-WorkingHours.pcap_ISCX.csv"
)

df = pd.read_csv(monday_file)

print("Original shape:", df.shape)

Original shape: (529918, 79)


In [4]:
# Remove leading and trailing spaces from column names

df.columns = df.columns.str.strip()

print("Cleaned column names:")
print(df.columns.tolist())

Cleaned column names:
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag C

In [5]:
print("Target column:", df.columns[-1])
print("\nLabels:")
print(df["Label"].value_counts())

Target column: Label

Labels:
Label
BENIGN    529918
Name: count, dtype: int64


In [6]:
# Check missing values

missing_values = df.isnull().sum()

print("Total missing values:", missing_values.sum())

print("\nColumns with missing values:")
print(missing_values[missing_values > 0])

Total missing values: 64

Columns with missing values:
Flow Bytes/s    64
dtype: int64


In [7]:
# Check infinite values

numeric_columns = df.select_dtypes(include=np.number).columns

infinite_values = np.isinf(df[numeric_columns]).sum()

print("Total infinite values:", infinite_values.sum())

print("\nColumns with infinite values:")
print(infinite_values[infinite_values > 0])

Total infinite values: 810

Columns with infinite values:
Flow Bytes/s      373
Flow Packets/s    437
dtype: int64


In [8]:
# Check duplicate rows

duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 26935


In [9]:
# Replace infinite values with NaN

df.replace([np.inf, -np.inf], np.nan, inplace=True)

print("Missing values after converting infinity to NaN:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Missing values after converting infinity to NaN:
Flow Bytes/s      437
Flow Packets/s    437
dtype: int64


In [10]:
# Remove rows containing missing values

before = len(df)

df.dropna(inplace=True)

after = len(df)

print("Rows before cleaning:", before)
print("Rows after removing missing values:", after)
print("Rows removed:", before - after)

Rows before cleaning: 529918
Rows after removing missing values: 529481
Rows removed: 437


In [11]:
# Remove duplicate rows

before = len(df)

df.drop_duplicates(inplace=True)

after = len(df)

print("Rows before removing duplicates:", before)
print("Rows after removing duplicates:", after)
print("Duplicate rows removed:", before - after)

Rows before removing duplicates: 529481
Rows after removing duplicates: 502650
Duplicate rows removed: 26831


In [12]:
# Verify that the dataset is clean

print("Final shape:", df.shape)

print("\nMissing values:", df.isnull().sum().sum())

numeric_columns = df.select_dtypes(include=np.number).columns

print("Infinite values:", np.isinf(df[numeric_columns]).sum().sum())

print("Duplicate rows:", df.duplicated().sum())

Final shape: (502650, 79)

Missing values: 0
Infinite values: 0
Duplicate rows: 0


In [13]:
# Create folder for processed CICIDS2017 data

processed_path = "../data/CICIDS2017/processed"

os.makedirs(processed_path, exist_ok=True)

print("Processed data folder ready:")
print(processed_path)

Processed data folder ready:
../data/CICIDS2017/processed


In [14]:


cleaning_summary = []

for file in cic_files:
    
    print("\nProcessing:", os.path.basename(file))
    
    # Load dataset
    data = pd.read_csv(file)
    
    original_rows = len(data)
    
    # Clean column names
    data.columns = data.columns.str.strip()
    
    # Convert infinite values to NaN
    data.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    # Remove missing values
    missing_removed = data.isnull().any(axis=1).sum()
    data.dropna(inplace=True)
    
    # Remove duplicates
    duplicate_removed = data.duplicated().sum()
    data.drop_duplicates(inplace=True)
    
    # Save cleaned file
    output_file = os.path.join(
        processed_path,
        os.path.basename(file)
    )
    
    data.to_csv(output_file, index=False)
    
    cleaning_summary.append({
        "File": os.path.basename(file),
        "Original Rows": original_rows,
        "Final Rows": len(data),
        "Missing Rows Removed": missing_removed,
        "Duplicate Rows Removed": duplicate_removed
    })
    
    print("Original rows:", original_rows)
    print("Final rows:", len(data))
    print("Saved:", output_file)

print("\nAll files cleaned successfully!")


Processing: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Original rows: 225745
Final rows: 223082
Saved: ../data/CICIDS2017/processed\Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv

Processing: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Original rows: 286467
Final rows: 213777
Saved: ../data/CICIDS2017/processed\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv

Processing: Friday-WorkingHours-Morning.pcap_ISCX.csv
Original rows: 191033
Final rows: 184044
Saved: ../data/CICIDS2017/processed\Friday-WorkingHours-Morning.pcap_ISCX.csv

Processing: Monday-WorkingHours.pcap_ISCX.csv
Original rows: 529918
Final rows: 502650
Saved: ../data/CICIDS2017/processed\Monday-WorkingHours.pcap_ISCX.csv

Processing: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Original rows: 288602
Final rows: 252790
Saved: ../data/CICIDS2017/processed\Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv

Processing: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Origin

In [15]:
cleaning_summary_df = pd.DataFrame(cleaning_summary)

display(cleaning_summary_df)

,File,Original Rows,Final Rows,Missing Rows Removed,Duplicate Rows Removed
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,223082,34,2629
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,213777,371,72319
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,184044,122,6867
3,Monday-WorkingHours.pcap_ISCX.csv,529918,502650,437,26831
4,Thursday-WorkingHours-Afternoon-Infilteration....,288602,252790,207,35605
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,170366,164179,135,6052
6,Tuesday-WorkingHours.pcap_ISCX.csv,445909,421626,264,24019
7,Wednesday-workingHours.pcap_ISCX.csv,692703,610492,1297,80914
